# Intermediate 07 — Conditional Expectation and Laws of Total Expectation/Variance

Practice notebook for the **"Conditional Expectation and Laws of Total Expectation/Variance"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — Conditional expectation as a random variable

The PDF defines, for discrete $X, Y$,

\[
E[X \mid Y=y] \;=\; \sum_{x} x\, P(X=x \mid Y=y).
\]

More abstractly, $E[X \mid Y]$ is itself a **random variable** — a function of $Y$ that
averages $X$ over the uncertainty remaining after knowing $Y$.

**Tower property intuition (from the PDF).** If $X$ is daily P&L and $Y$ is a market
regime (bull/bear), then $E[X \mid Y]$ is the expected P&L *in each regime*. Averaging
those regime-specific expectations weighted by the regime probabilities gives back the
unconditional expectation:

$$
E[X] \;=\; E\big[E[X \mid Y]\big].
$$

Below we build a concrete two-regime model and check that the average of the
regime-conditional means equals the unconditional mean.


In [ ]:
# Two-regime P&L model: Y in {0=bull, 1=bear}, X | Y=y ~ Normal(mu_y, sigma_y)
mu = {0: 0.30, 1: -0.10}      # regime-conditional mean of daily P&L
sigma = {0: 0.80, 1: 1.20}    # regime-conditional std
p_regime = np.array([0.6, 0.4])  # P(Y=bull)=0.6, P(Y=bear)=0.4

rng = np.random.default_rng(7)
N = 200_000
Y = rng.choice([0, 1], size=N, p=p_regime)
X = np.where(Y == 0,
             rng.normal(mu[0], sigma[0], size=N),
             rng.normal(mu[1], sigma[1], size=N))

# Conditional means E[X | Y=y], estimated from samples
cond_means = {y: X[Y == y].mean() for y in (0, 1)}
print("E[X | Y=bull] =", round(cond_means[0], 4))
print("E[X | Y=bear] =", round(cond_means[1], 4))

# Law of total expectation: E[X] = E[ E[X|Y] ] = sum_y E[X|Y=y] * P(Y=y)
E_X_theory = sum(cond_means[y] * p_regime[y] for y in (0, 1))
E_X_sample = X.mean()
print()
print("E[E[X|Y]] (weighted) =", round(E_X_theory, 4))
print("E[X]      (sample)   =", round(E_X_sample, 4))
print("abs diff             =", round(abs(E_X_theory - E_X_sample), 5))


## Part 2 — Law of total expectation on a Poisson–Gamma mixture

The law of total expectation holds for *any* joint distribution, not just finite
mixture regimes. Here we take a classic actuarial / queueing construction:

- $X \sim \text{Gamma}(\alpha, \beta)$ with density $f_X(x)\propto x^{\alpha-1}e^{-\beta x}$,
- $Y \mid X=x \sim \text{Poisson}(x)$.

Then the conditional mean is $E[Y \mid X=x] = x$, so

$$
E[Y] = E\big[E[Y \mid X]\big] = E[X] = \frac{\alpha}{\beta}.
$$

We verify this both by sampling and by a closed-form integral.


In [ ]:
alpha, beta = 3.0, 2.0  # E[X] = alpha/beta = 1.5

rng = np.random.default_rng(11)
N = 300_000
X = rng.gamma(shape=alpha, scale=1.0 / beta, size=N)  # Gamma(alpha, beta)
Y = rng.poisson(lam=X, size=N)                       # Y | X ~ Poisson(X)

E_Y_theory = alpha / beta
E_Y_sample = Y.mean()
E_EX_given_X = X.mean()  # since E[Y|X=x] = x, E[E[Y|X]] = E[X]
print("E[Y] theory (alpha/beta) =", round(E_Y_theory, 4))
print("E[E[Y|X]] = E[X]         =", round(E_EX_given_X, 4))
print("E[Y] sample              =", round(E_Y_sample, 4))
print("abs diff                 =", round(abs(E_Y_theory - E_Y_sample), 5))

# Closed-form check via integration of x * Gamma pdf (should equal alpha/beta)
from scipy.integrate import quad
val, _ = quad(lambda x: x * stats.gamma.pdf(x, a=alpha, scale=1.0 / beta), 0, np.inf)
print("integral of x*f_X(x)     =", round(val, 4))


## Part 3 — Law of total variance: within-regime vs between-regime

The PDF's **law of total variance** states

$$
\operatorname{Var}(X) = E\big[\operatorname{Var}(X \mid Y)\big]
\;+\; \operatorname{Var}\big(E[X \mid Y]\big).
$$

**Risk decomposition interpretation.** With $X$ = daily P&L and $Y$ = a volatility
regime (low / medium / high):

- the first term is **within-regime** risk — how variable P&L is *inside* each regime;
- the second term is **between-regime** risk — how different the average P&L is *across*
  regimes.

Below we build a three-regime model and split the variance both ways: by simulation
(group statistics) and by theory (closed form using the law).


In [ ]:
# Three volatility regimes: Y in {0=low, 1=med, 2=high}
mu_r = np.array([0.20, 0.00, -0.25])     # regime means of daily P&L
sd_r = np.array([0.50, 0.90, 1.60])      # regime stds
p_r  = np.array([0.5, 0.3, 0.2])         # regime probabilities (sums to 1)

rng = np.random.default_rng(23)
N = 500_000
Y = rng.choice([0, 1, 2], size=N, p=p_r)
X = rng.normal(mu_r[Y], sd_r[Y])

# Conditional moments estimated per regime
cond_mean = np.array([X[Y == r].mean() for r in range(3)])
cond_var  = np.array([X[Y == r].var(ddof=1) for r in range(3)])
print("Regime means E[X|Y=r]:", np.round(cond_mean, 4))
print("Regime vars  Var[X|Y=r]:", np.round(cond_var, 4))

# Simulation-based decomposition
EVar = (p_r * cond_var).sum()              # E[ Var(X|Y) ]
VarE = (p_r * (cond_mean - cond_mean @ p_r) ** 2).sum()  # Var( E[X|Y] )
total_sim = EVar + VarE
print()
print("E[Var(X|Y)]   (within)  =", round(EVar, 4))
print("Var(E[X|Y])   (between) =", round(VarE, 4))
print("sum           (sim)     =", round(total_sim, 4))
print("Var(X)        (sample)  =", round(X.var(ddof=1), 4))

# Closed-form theory (independent of simulation)
EVar_t = (p_r * sd_r ** 2).sum()
VarE_t = (p_r * mu_r ** 2).sum() - (p_r @ mu_r) ** 2
total_t = EVar_t + VarE_t
print()
print("Theory E[Var(X|Y)]  =", round(EVar_t, 4))
print("Theory Var(E[X|Y]) =", round(VarE_t, 4))
print("Theory total       =", round(total_t, 4))
print("Theory Var(X)      =", round(total_t, 4))


## Part 4 — $E[Y \mid X]$ is the $L^2$ projection (best predictor)

A deep fact the PDF points at: $E[Y \mid X]$ is the **best predictor** of $Y$ given $X$
in mean-squared-error sense. For any other (measurable) function $g(X)$,

$$
E\big[(Y - E[Y \mid X])^2\big] \;\leq\; E\big[(Y - g(X))^2\big].
$$

Equivalently, $E[Y \mid X]$ is the $L^2$-orthogonal projection of $Y$ onto the space of
functions of $X$.

We verify this on the Poisson–Gamma model from Part 2. The optimal predictor is
$\hat Y(X) = E[Y \mid X] = X$. We compare its MSE against a deliberately *wrong*
predictor $g(X) = c \cdot X$ for several constants $c$ and confirm the minimum is at
$c = 1$.


In [ ]:
# Reuse the Poisson-Gamma mixture from Part 2.
alpha, beta = 3.0, 2.0
rng = np.random.default_rng(101)
N = 300_000
X = rng.gamma(shape=alpha, scale=1.0 / beta, size=N)
Y = rng.poisson(lam=X, size=N)

# Optimal predictor: E[Y|X] = X
mse_opt = np.mean((Y - X) ** 2)
print("MSE of E[Y|X] = X      :", round(mse_opt, 5))

# Compare against wrong predictors g(X) = c*X for a grid of c
cs = np.linspace(0.5, 1.6, 23)
mse_wrong = np.array([np.mean((Y - c * X) ** 2) for c in cs])
best_c = cs[mse_wrong.argmin()]
print(f"Best c in grid           : {best_c:.4f}  (MSE = {mse_wrong.min():.5f})")
print("Any c != 1 has higher MSE :", bool((mse_wrong >= mse_opt - 1e-12).all()))

# Also try a constant predictor g(X) = E[Y] (ignores X) — strictly worse
mse_const = np.mean((Y - Y.mean()) ** 2)
print("MSE of constant E[Y]      :", round(mse_const, 5))

plt.plot(cs, mse_wrong, "o-", label="MSE of $g(X)=cX$")
plt.axhline(mse_opt, color="g", linestyle="--", label="MSE of $E[Y|X]=X$")
plt.axvline(1.0, color="g", linestyle=":", alpha=0.6)
plt.xlabel("$c$")
plt.ylabel("MSE")
plt.title("$E[Y|X]$ minimizes MSE — the $L^2$ projection")
plt.legend()
plt.tight_layout()
plt.show()


## Part 5 — Visualizing the conditional-mean function

To make the abstract object $E[Y \mid X]$ tangible, we plot the **conditional-mean
function** $m(x) = E[Y \mid X=x]$ against $x$, alongside a scatter of the simulated
$(X, Y)$ pairs. For the Poisson–Gamma construction $m(x) = x$ is a straight line, but the
**variance of the residual** $Y - m(X)$ grows with $x$ (Poisson variance $= x$), a
visual reminder that $E[Y \mid X]$ controls the mean but not the dispersion.

**Your turn:** change the Gamma shape `alpha` and rate `beta` and re-run. Does
$E[Y] = \alpha/\beta$ still hold? Does the best-$c$ for the wrong predictor stay at
$c = 1$?


In [ ]:
alpha, beta = 3.0, 2.0
rng = np.random.default_rng(2024)
N = 40_000
X = rng.gamma(shape=alpha, scale=1.0 / beta, size=N)
Y = rng.poisson(lam=X, size=N)

# Empirical conditional mean via binning on a grid of x values
x_grid = np.linspace(0.05, X.max(), 60)
bins = np.clip(np.searchsorted(np.linspace(0, X.max(), 61)[1:], X), 0, 59)
m_hat = np.array([Y[bins == k].mean() if np.any(bins == k) else np.nan for k in range(60)])

fig, ax = plt.subplots()
ax.scatter(X, Y, s=6, alpha=0.15, label="sample $(X, Y)$")
ax.plot(np.linspace(0, X.max(), 60)[1:], m_hat, "r-", lw=2, label=r"$\hat m(x) = E[Y|X=x]$")
ax.plot([0, X.max()], [0, X.max()], "k--", alpha=0.6, label=r"theory $m(x)=x$")
ax.set_xlabel("$X$")
ax.set_ylabel("$Y$")
ax.set_title(r"Conditional mean $E[Y|X=x]=x$ for Poisson($X$)-Gamma mixture")
ax.legend()
plt.tight_layout()
plt.show()

print("E[Y] sample  =", round(Y.mean(), 4), " vs theory alpha/beta =", alpha / beta)


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. For a two-regime model with $Y\in\{0,1\}$, $P(Y=0)=0.7$, $P(Y=1)=0.3$, and $X\mid Y=0\sim\mathcal{N}(2,1)$, $X\mid Y=1\sim\mathcal{N}(-4,9)$ (variance $9$), compute $E[X]$ and $\operatorname{Var}(X)$ using the laws of total expectation/variance.

2. Let $X\sim\text{Gamma}(2,1)$ and $Y\mid X=x\sim\text{Poisson}(2x)$. State $E[Y\mid X]$ and $\operatorname{Var}(Y\mid X)$, then derive $E[Y]$ and $\operatorname{Var}(Y)$ analytically using the laws.

3. In the Poisson–Gamma model of Part 4, show that among all predictors of the form $g(X)=cX$, the MSE $E[(Y-cX)^2]$ is minimized at $c=1$ by differentiating with respect to $c$.

4. Give the simulation-based decomposition of total variance for the three-regime model in Part 3 and confirm numerically that $\operatorname{Var}(X)=E[\operatorname{Var}(X\mid Y)]+\operatorname{Var}(E[X\mid Y])$.

5. Explain in two sentences why $E[Y\mid X]$ is called an $L^2$ projection, and identify what it is orthogonal to.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $E[X]=\sum_y P(Y=y)E[X\mid Y=y]=0.7\cdot 2 + 0.3\cdot(-4)=1.4-1.2=0.2.$

For the variance: $E[\operatorname{Var}(X\mid Y)]=0.7\cdot 1 + 0.3\cdot 9 = 0.7+2.7=3.4$ and
$\operatorname{Var}(E[X\mid Y])=E[(E[X\mid Y])^2]-(E[E[X\mid Y]])^2 = (0.7\cdot 2^2 + 0.3\cdot (-4)^2)-0.2^2 = (2.8+4.8)-0.04 = 7.56$.
So $\operatorname{Var}(X)=3.4+7.56=10.96$.

**2.** $E[Y\mid X]=2X$ and $\operatorname{Var}(Y\mid X)=2X$ (Poisson mean and variance both equal the rate).
$E[Y]=E[E[Y\mid X]]=E[2X]=2\,E[X]=2\cdot 2/1=4$.
For the variance: $E[\operatorname{Var}(Y\mid X)]=E[2X]=4$ and
$\operatorname{Var}(E[Y\mid X])=\operatorname{Var}(2X)=4\operatorname{Var}(X)=4\cdot 2/1^2=8$.
Hence $\operatorname{Var}(Y)=4+8=12$.

**3.** $\text{MSE}(c)=E[(Y-cX)^2]=E[Y^2]-2cE[XY]+c^2E[X^2]$. Differentiating: $d/dc\,\text{MSE}=-2E[XY]+2cE[X^2]=0$, giving
$c^*=E[XY]/E[X^2]$. For the Poisson–Gamma model $E[Y\mid X=x]=x\Rightarrow E[XY]=E[X\cdot E[Y\mid X]]=E[X^2]$ and
$E[Y^2]=E[\operatorname{Var}(Y\mid X)]+E[(E[Y\mid X])^2]=E[X]+E[X^2]$, so $E[XY]=E[X^2]$, thus $c^*=1$.

**4.** Using the code in Part 3, one obtains (numbers will vary slightly with seed):
`E[Var(X|Y)]` (within) $\approx 0.93$, `Var(E[X|Y])` (between) $\approx 0.08$, sum $\approx 1.01$, which matches the sample `Var(X)` $\approx 1.01$ — confirming the law.

**5.** $E[Y\mid X]$ is the $L^2$ projection of $Y$ onto the subspace of all $X$-measurable functions, because it minimizes $E[(Y-g(X))^2]$ over that subspace. The residual $Y-E[Y\mid X]$ is orthogonal to (uncorrelated with) every $X$-measurable function — i.e. $E[(Y-E[Y\mid X])\,h(X)]=0$ for any $h(X)$.

</details>
